# Study 882 — Gas-Price → Discretionary — the teardown

The predictive-regression slope with a Newey-West HAC *t*, the R², the tercile cross-check, the parallel energy-tilt regression, the 2,000-permutation placebo, the two-era robustness cut, the costed spread timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2005-02-28', 'end': '2026-05-31', 'n_months': 256, 'gas': 'RB=F', 'fingerprint': 'c131f45ae347', 'beta': -0.0171, 't_nw': -0.71, 't_ols': -0.79, 'r2_pct': 0.25, 'alpha_pct': 0.26, 'fwd_down_pct': 0.19, 'fwd_up_pct': -0.08, 'welch_t': -0.4, 'enr_beta': -0.0045, 'enr_t': -0.11, 'enr_r2': 0.01, 'enr_fwd_down_pct': 0.21, 'enr_fwd_up_pct': 0.56, 'enr_welch_t': 0.35, 'placebo_obs': -0.0171, 'placebo_mean': 0.0004, 'placebo_sd': 0.0218, 'placebo_p': 0.431, 'placebo_draws': 2000, 'era1_n': 131, 'era1_beta': -0.0054, 'era1_t': -0.16, 'era1_r2': 0.03, 'era2_n': 125, 'era2_beta': -0.0289, 'era2_t': -0.88, 'era2_r2': 0.57, 'ls1_gross': 0.089, 'ls1_net': 0.028, 'ls1_t': 0.1, 'ls1_sharpe': 0.02, 'ls1_ann': 0.3, 'ls5_net': -0.046, 'ls5_t': -0.16, 'ls5_sharpe': -0.04, 'hit': 0.504, 'null_mean_t': -0.61, 'null_sd_t': 1.0, 'null_fire': 1, 'planted_beta': -0.1294, 'planted_t': -5.92, 'planted_r2': 10.44}

## The headline — predictive regression  `r_(XLY-XLP)[t+1] = a + b·r_gas[t]`

Monthly, one documented lag: gas return known at the close of month `t`, the XLY−XLP spread return realised over month `t+1`. Slope `b` is the pump-tax coefficient (expected < 0).

In [2]:
print(f"slope beta   : {R['beta']:+.4f}   NW(6) t = {R['t_nw']:+.2f}   "
      f"OLS t = {R['t_ols']:+.2f}   R2 = {R['r2_pct']:.2f}%")
print(f"alpha        : {R['alpha_pct']:+.2f}%/mo   n = {R['n_months']} months")
print(f"tercile check: fwd XLY-XLP after gas-down {R['fwd_down_pct']:+.2f}% vs "
      f"after gas-up {R['fwd_up_pct']:+.2f}% (Welch t = {R['welch_t']:+.2f})")
print('  claim: beta < 0 (gas up -> discretionary lags staples). Found: right sign, but ~0.')

slope beta   : -0.0171   NW(6) t = -0.71   OLS t = -0.79   R2 = 0.25%
alpha        : +0.26%/mo   n = 256 months
tercile check: fwd XLY-XLP after gas-down +0.19% vs after gas-up -0.08% (Welch t = -0.40)
  claim: beta < 0 (gas up -> discretionary lags staples). Found: right sign, but ~0.


## The energy tilt — `r_(XLE-SPY)[t+1] = a + b·r_gas[t]`  (claim: b > 0)

In [3]:
print(f"slope beta   : {R['enr_beta']:+.4f}   NW(6) t = {R['enr_t']:+.2f}   R2 = {R['enr_r2']:.2f}%")
print(f"tercile check: fwd XLE-SPY after gas-down {R['enr_fwd_down_pct']:+.2f}% vs "
      f"after gas-up {R['enr_fwd_up_pct']:+.2f}% (Welch t = {R['enr_welch_t']:+.2f})")
print('  the tercile leans right (energy beats after dear gas) but the slope is a flat zero.')

slope beta   : -0.0045   NW(6) t = -0.11   R2 = 0.01%
tercile check: fwd XLE-SPY after gas-down +0.21% vs after gas-up +0.56% (Welch t = +0.35)
  the tercile leans right (energy beats after dear gas) but the slope is a flat zero.


## Placebo — permute the target, keep the predictor (2,000 draws)

In [4]:
print(f"observed beta {R['placebo_obs']:+.4f} vs placebo mean {R['placebo_mean']:+.4f} "
      f"(sd {R['placebo_sd']:.4f}) -> two-sided p = {R['placebo_p']:.3f}")

observed beta -0.0171 vs placebo mean +0.0004 (sd 0.0218) -> two-sided p = 0.431


## Robustness — two eras (split 2016-01-01)

In [5]:
print(f"2005-2015 (n={R['era1_n']}): beta {R['era1_beta']:+.4f}  NW t = {R['era1_t']:+.2f}  R2 = {R['era1_r2']:.2f}%")
print(f"2016-2026 (n={R['era2_n']}): beta {R['era2_beta']:+.4f}  NW t = {R['era2_t']:+.2f}  R2 = {R['era2_r2']:.2f}%")
print('  the sign is at least stable (negative in both) -- but insignificant in both. A stable nothing.')

2005-2015 (n=131): beta -0.0054  NW t = -0.16  R2 = 0.03%
2016-2026 (n=125): beta -0.0289  NW t = -0.88  R2 = 0.57%
  the sign is at least stable (negative in both) -- but insignificant in both. A stable nothing.


## The timer — can you get paid for it?

Trade `-sign(gas_ret[t])` of the XLY−XLP spread next month (a 2x NAV long/short book); one-way cost × NAV per rebalance leg on both legs, 50 bps/yr borrow on the short leg.

In [6]:
print(f"spread 1bp: gross {R['ls1_gross']:+.3f}%/mo -> net {R['ls1_net']:+.3f}%/mo "
      f"(t={R['ls1_t']:+.2f}, Sharpe {R['ls1_sharpe']:.2f}, ~{R['ls1_ann']:+.1f}%/yr, hit {R['hit']:.3f})")
print(f"spread 5bp: net {R['ls5_net']:+.3f}%/mo (t={R['ls5_t']:+.2f}, Sharpe {R['ls5_sharpe']:.2f})")
print('  a coin flip (hit 0.504); flat-to-negative once you pay realistic two-leg costs.')

spread 1bp: gross +0.089%/mo -> net +0.028%/mo (t=+0.10, Sharpe 0.02, ~+0.3%/yr, hit 0.504)
spread 5bp: net -0.046%/mo (t=-0.16, Sharpe -0.04)
  a coin flip (hit 0.504); flat-to-negative once you pay realistic two-leg costs.


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted negative slope.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from gas_discretionary import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_series(edge=0.0, seed=882+s))['t_nw'] for s in range(20)])
print(f"null (edge=0), 20 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), "
      f"|t|>=2 in {(abs(null_t)>=2).sum()}/20")
planted = st.synthetic_detect(data.synthetic_series(edge=0.35, seed=882))
print(f"planted (edge=0.35): beta = {planted['beta']:+.4f}, NW t = {planted['t_nw']:+.2f}, R2 = {planted['r2_pct']:.2f}%")

null (edge=0), 20 seeds: NW t mean -0.61 (sd 1.00), |t|>=2 in 1/20
planted (edge=0.35): beta = -0.1294, NW t = -5.92, R2 = 10.44%


## Verdict

- **Signal — None.** The pump-tax rotation does **not** replicate on 2005–2026 US ETFs: the slope is **-0.0171** (NW *t* = **-0.71**, R² = **0.25%**) — the *right* (negative) sign, but a flat, insignificant nothing. It is p = 0.43 in a 2,000-draw placebo and stays tiny and insignificant across both eras (*t* = -0.16 / -0.88); the energy tilt is flatter still (*t* = -0.11). The 20-seed synthetic control recovers a *planted* slope cleanly (*t* = -5.92), so the flat real-tape result is a genuine null, not a bug.
- **Tradability — Mirage.** The spread timer is a coin flip (hit 0.504): gross +0.089%/mo, net +0.028%/mo at 1 bp and -0.046%/mo once you pay realistic two-leg costs (Sharpe ≈ 0). No paycheck here.